# 10 ações diversificam de verdade?

**Python Finance Lab · Aula 5** · *Finanças na prática, notebook por notebook.*

---

## Parte 1 · A pergunta

Na aula 2, apareceu um fato curioso: a carteira oscilou menos do que a maioria das ações que a compõem. Na aula 4, vimos que o rebalanceamento só tem trabalho quando as ações se afastam umas das outras. As duas observações têm a mesma explicação: as ações não sobem e caem todas juntas, nem na mesma intensidade. Quando uma cai, outra pode estar subindo, e as oscilações se compensam em parte.

O nome desse efeito é **diversificação**, e a medida de quanto duas ações andam juntas é a **correlação**. Nesta aula, vamos responder a duas perguntas. Quanto as nossas 10 ações se movem juntas? E quanto risco cai a cada ação acrescentada à carteira, de 1 até 10?

Para responder à segunda, vamos testar todas as carteiras possíveis com as 10 ações: são 1.023 combinações. Do lado do Python, quem faz esse trabalho é a função `combinations`, que lista todas as combinações de um grupo, e o laço dentro de outro laço.

*Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*

## Parte 2 · Conceitos

### Bloco 2 · O vocabulário da aula

**Covariância.** Mede se duas séries de retornos costumam ficar do mesmo lado da média ao mesmo tempo. Positiva: quando uma está acima da média, a outra também tende a estar. Negativa: tendem a ficar em lados opostos. O problema da covariância é a escala: ela depende do tamanho das oscilações, o que dificulta comparar pares diferentes.

**Correlação.** É a covariância "padronizada", dividida pelas volatilidades das duas séries. Fica sempre entre −1 e +1:

- **+1:** as duas sobem e descem sempre juntas, na mesma proporção;
- **0:** os movimentos de uma não dizem nada sobre os da outra;
- **−1:** quando uma sobe, a outra desce, sempre.

Entre ações de um mesmo país, a correlação costuma ser positiva, porque todas sentem as notícias da economia. A pergunta é *quão* positiva.

**Risco específico.** É a parte da oscilação que vem de cada empresa: um balanço ruim, uma troca de diretoria, um processo na justiça. Ele é diversificável: em uma carteira com muitas empresas, os sustos de uma são diluídos pelas outras.

**Risco de mercado.** É a parte que atinge todas as empresas ao mesmo tempo: juros, inflação, eleições, crises globais. Ele não é diversificável: não importa quantas ações brasileiras a carteira tenha, uma crise no país atinge todas.

**Diversificação.** Combinar ativos que não andam perfeitamente juntos para reduzir o risco sem, necessariamente, reduzir o retorno esperado. Ela elimina o risco específico, mas não o de mercado.

**Setores que andam juntos.** Empresas expostas aos mesmos fatores tendem a ter correlação mais alta. Exportadoras de commodities, por exemplo, reagem juntas ao dólar e aos preços internacionais.

**Correlação não é causa.** Duas ações correlacionadas não se influenciam necessariamente; muitas vezes, as duas reagem à mesma notícia.

## Parte 3 · Fórmulas

### Bloco 3 · As contas da aula

**Correlação** entre os retornos das ações $i$ e $j$:

$$\rho_{ij} = \frac{\text{cov}(r_i, r_j)}{\sigma_i\,\sigma_j}$$

**Variância de uma carteira com pesos iguais**, com $N$ ações:

$$\sigma_p^2 = \frac{1}{N}\,\overline{\sigma^2} + \left(1 - \frac{1}{N}\right)\overline{\text{cov}}$$

em que $\overline{\sigma^2}$ é a variância média das ações e $\overline{\text{cov}}$ é a covariância média entre os pares. Quando $N$ cresce, o primeiro termo encolhe até sumir, e a variância da carteira se aproxima de $\overline{\text{cov}}$: o risco de mercado, o piso que a diversificação não consegue furar.

*Exemplo à mão:* duas ações com volatilidade de 30% ao ano cada, e a carteira com metade em cada uma. A covariância entre elas é $\rho \times 0{,}30 \times 0{,}30$.

| Correlação | Covariância | Variância da carteira | Volatilidade da carteira |
|---|---|---|---|
| +1 | 0,090 | $\tfrac12 \times 0{,}09 + \tfrac12 \times 0{,}090 = 0{,}0900$ | 30,0% |
| +0,5 | 0,045 | $\tfrac12 \times 0{,}09 + \tfrac12 \times 0{,}045 = 0{,}0675$ | 26,0% |
| 0 | 0 | $\tfrac12 \times 0{,}09 + 0 = 0{,}0450$ | 21,2% |
| −1 | −0,090 | $\tfrac12 \times 0{,}09 - \tfrac12 \times 0{,}090 = 0$ | 0,0% |

Com correlação +1, juntar as duas não reduz nada: o risco continua em 30%. Quanto menor a correlação, maior a redução. No extremo de −1, as oscilações se anulam por completo.

## Parte 4 · Implementação

### Bloco 4 · Importar as bibliotecas

As bibliotecas de sempre e uma nova: `combinations`, da `itertools`, uma biblioteca que já vem com o Python e reúne ferramentas para percorrer grupos de itens. O `from ... import ...` traz só a ferramenta que vamos usar, sem precisar escrever `itertools.` antes dela.

> **O que esperar:** nada. Se aparecer `ModuleNotFoundError`, instale com `%pip install yfinance pandas matplotlib requests` e reinicie o kernel (menu Kernel → Restart Kernel).

In [ ]:
from itertools import combinations

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

### Bloco 5 · As premissas

As mesmas ações e datas de sempre. O capital da aula 1 volta só para a comparação final, com a carteira das aulas anteriores.

> **O que esperar:** nada.

In [ ]:
ACOES = [
    "ITUB4.SA", "VALE3.SA", "PETR4.SA", "WEGE3.SA", "ABEV3.SA",
    "EGIE3.SA", "SUZB3.SA", "RADL3.SA", "VIVT3.SA", "TOTS3.SA",
]
DATA_INICIO = "2021-10-01"
DATA_FIM = "2026-09-30"
CAPITAL = 2000

### Bloco 6 · Os preços das 10 ações

Nesta aula, só precisamos das ações: nem o Ibovespa nem o CDI entram nas contas. O código é a primeira parte do bloco de dados das aulas anteriores.

> **O que esperar:** as três últimas linhas da tabela de preços, com as 10 ações em ordem alfabética.

In [ ]:
dados = yf.download(ACOES, start=DATA_INICIO, auto_adjust=True, progress=False)
precos = dados["Close"].loc[DATA_INICIO:DATA_FIM].ffill()
precos.columns = precos.columns.str.replace(".SA", "")
precos.tail(3)

### Bloco 7 · Os retornos diários

Correlação se mede nos retornos, e não nos preços. Dois preços que sobem ao longo dos anos parecem andar juntos num gráfico, mesmo que, no dia a dia, um suba quando o outro cai. O que interessa à diversificação é o movimento de cada dia.

O cálculo é o `pct_change` da aula 2, agora na tabela das 10 ações de uma vez.

> **O que esperar:** o tamanho da tabela, com uma linha a menos que a de preços, e os retornos dos cinco primeiros dias, em %.

In [ ]:
retornos = precos.pct_change().dropna()
print("Linhas e colunas:", retornos.shape)
(retornos * 100).head()

### Bloco 8 · A matriz de correlação

O método `corr()` calcula a correlação entre cada par de colunas e devolve uma tabela 10×10: a **matriz de correlação**. A linha do Itaú com a coluna da Vale mostra a correlação entre os dois.

Dois detalhes que valem para qualquer matriz de correlação:

- **A diagonal vale 1:** cada ação tem correlação perfeita consigo mesma.
- **A matriz é simétrica:** a correlação do Itaú com a Vale é a mesma da Vale com o Itaú. Metade da tabela repete a outra metade.

> **O que esperar:** uma tabela 10×10 com 1 na diagonal e números entre 0 e 1 no resto, quase todos positivos.

In [ ]:
corr = retornos.corr()
corr.round(2)

**Depois de rodar.** **[Preencher: a faixa de valores encontrada.]**

### Bloco 9 · O mapa de calor

Uma tabela com 100 números é difícil de ler. O mapa de calor pinta cada célula com uma cor que representa o valor.

- `plt.subplots` cria a figura (`fig`) e o eixo (`ax`) antes de desenhar. Até aqui, o `.plot` do pandas fazia isso por nós; agora, precisamos do eixo para desenhar a imagem e escrever os números.
- `ax.imshow(corr, ...)` desenha a tabela como imagem. A escala de cores `"RdBu_r"` vai do azul (−1) ao vermelho (+1), passando pelo branco (0). Fixar `vmin=-1` e `vmax=1` garante que o branco fique exatamente no zero.
- `range(10)` gera os números de 0 a 9, as posições das linhas e das colunas. `set_xticks` e `set_yticks` colocam os nomes das ações nessas posições.
- O **laço dentro de outro laço** percorre todas as células: para cada linha `i`, percorre cada coluna `j`. São 10 × 10 = 100 voltas. `corr.iloc[i, j]` pega o valor da linha `i` e da coluna `j` pela posição.
- O `if` da aula 4 escolhe a cor do texto: branco nas células escuras, preto nas claras.

> **O que esperar:** um quadrado 10×10 em tons de vermelho, mais escuro na diagonal, com o valor escrito em cada célula. A barra ao lado traduz as cores em números. Se aparecer alguma célula azulada, aquele par anda em sentidos opostos.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
imagem = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)

nomes = list(corr.columns)
ax.set_xticks(range(len(nomes)), nomes, rotation=45, ha="right")
ax.set_yticks(range(len(nomes)), nomes)

for i in range(len(nomes)):
    for j in range(len(nomes)):
        valor = corr.iloc[i, j]
        if abs(valor) > 0.6:
            cor_texto = "white"
        else:
            cor_texto = "black"
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=8, color=cor_texto)

fig.colorbar(imagem, ax=ax, label="Correlação")
ax.set_title("Correlação entre os retornos diários das 10 ações")
plt.show()

**Depois de rodar.** **[Preencher: quais blocos de cor mais forte aparecem fora da diagonal.]**

### Bloco 10 · Os pares que mais e menos andam juntos

Para listar os pares, usamos `combinations(grupo, 2)`, que devolve todas as duplas possíveis de um grupo, sem repetir e sem se importar com a ordem. Com três letras, por exemplo, ela devolve (A, B), (A, C) e (B, C). Cada dupla vem entre parênteses: é uma **tupla**, uma espécie de lista que não pode ser alterada.

No laço, `for a, b in combinations(...)` separa cada dupla em dois nomes, como o `.items()` da aula 4. Para cada par, guardamos os dois nomes e a correlação numa lista, que depois vira uma tabela ordenada da menor para a maior correlação.

Com 10 ações, são 45 pares: cada uma das 10 se combina com as outras 9, e cada par aparece duas vezes nessa conta, então $10 \times 9 / 2 = 45$.

O `to_string(index=False)` imprime a tabela sem a coluna de numeração das linhas.

> **O que esperar:** `[('A', 'B'), ('A', 'C'), ('B', 'C')]`, `Número de pares: 45`, a correlação média e as duas listas de três pares.

In [ ]:
print(list(combinations(["A", "B", "C"], 2)))
print()

pares = []
for a, b in combinations(nomes, 2):
    pares.append([a, b, corr.loc[a, b]])

pares = pd.DataFrame(pares, columns=["Ação 1", "Ação 2", "Correlação"])
pares = pares.sort_values("Correlação")

print("Número de pares:", len(pares))
print("Correlação média:", round(pares["Correlação"].mean(), 2))
print()
print("Os três pares que menos andam juntos:")
print(pares.head(3).round(2).to_string(index=False))
print()
print("Os três pares que mais andam juntos:")
print(pares.tail(3).round(2).to_string(index=False))

**Depois de rodar.** **[Preencher: a correlação média e os pares extremos. Os pares mais correlacionados são do mesmo setor ou expostos ao mesmo fator?]**

### Bloco 11 · A volatilidade de um grupo de ações

Agora, a segunda pergunta: quanto risco cai a cada ação acrescentada? Precisamos de uma função que calcule a volatilidade de uma carteira com pesos iguais, para qualquer grupo de ações.

- `retornos[list(grupo)]` seleciona as colunas do grupo. O `list` transforma a tupla que vem do `combinations` numa lista, que é o que o pandas espera para selecionar colunas.
- `.mean(axis=1)` é o retorno da carteira em cada dia: a média das ações do grupo.
- O resto é a volatilidade anual da aula 2.

Um detalhe: tirar a média dos retornos de cada dia equivale a uma carteira que **volta aos pesos iguais todo dia**, um rebalanceamento diário. É uma simplificação comum para estudar risco, e a validação do bloco 17 mostra que ela fica perto da carteira sem rebalanceamento das aulas anteriores.

O teste é simples: um "grupo" com uma ação só tem a volatilidade dessa ação.

> **O que esperar:** dois números iguais: a volatilidade anual do Itaú, calculada pelos dois caminhos.

In [ ]:
def volatilidade_grupo(retornos, grupo):
    """Volatilidade anual de uma carteira com pesos iguais nas ações do grupo."""
    retorno_carteira = retornos[list(grupo)].mean(axis=1)
    return retorno_carteira.std() * 252 ** 0.5


print(round(volatilidade_grupo(retornos, ["ITUB4"]), 6))
print(round(retornos["ITUB4"].std() * 252 ** 0.5, 6))      # esperado: o mesmo número

### Bloco 12 · Todas as 1.023 carteiras

Com 10 ações, existem 10 carteiras de uma ação, 45 de duas, 120 de três, e assim por diante, até uma única carteira com as 10. Somando, são 1.023 combinações. Vamos calcular a volatilidade de todas.

- O laço de fora percorre os tamanhos, de 1 a 10. `range(1, len(nomes) + 1)` vai de 1 até 10: o último número do `range` nunca entra, por isso o `+ 1`.
- O laço de dentro percorre todas as combinações daquele tamanho e guarda a volatilidade de cada uma, em %.
- No fim de cada tamanho, guardamos um resumo: quantas carteiras, a volatilidade média, a mínima, a máxima e a **típica**, que explicamos no próximo bloco.

> **O que esperar:** `Total de carteiras testadas: 1023` e uma tabela com 10 linhas. A volatilidade média cai rápido nas primeiras ações e cada vez mais devagar depois. A faixa entre a mínima e a máxima se estreita até virar um único valor, com as 10 ações.

In [ ]:
linhas = []
for k in range(1, len(nomes) + 1):
    vols = []
    for grupo in combinations(nomes, k):
        vols.append(volatilidade_grupo(retornos, grupo) * 100)
    vols = pd.Series(vols)
    linhas.append([k, len(vols), vols.mean(), vols.min(), vols.max(), ((vols ** 2).mean()) ** 0.5])

por_tamanho = pd.DataFrame(
    linhas, columns=["Ações", "Combinações", "Média (%)", "Mínima (%)", "Máxima (%)", "Típica (%)"]
).set_index("Ações")

print("Total de carteiras testadas:", por_tamanho["Combinações"].sum())
por_tamanho.round(2)

**Depois de rodar.** **[Preencher: a volatilidade média com 1, 3, 5 e 10 ações.]**

### Bloco 13 · A fórmula e o piso do risco

Agora, a fórmula do bloco 3. Precisamos de dois números:

- **A variância média das ações**, a média da diagonal da matriz de covariância. O `var()` dá a variância de cada coluna, e multiplicamos por 252 para levar à escala de um ano.
- **A covariância média entre os pares**, a média de tudo o que está fora da diagonal. A soma da matriz inteira, `cov.sum().sum()`, inclui a diagonal; tirando a soma das variâncias, sobra a soma das covariâncias, que dividimos pelo número de células fora da diagonal, $n(n - 1) = 90$.

A fórmula dá a variância **média** das carteiras de cada tamanho. Por isso, ela se compara à coluna "Típica", que é a raiz da média das variâncias, e não à coluna "Média", que é a média das volatilidades. As duas ficam muito próximas, mas não são iguais: tirar a raiz antes ou depois da média muda um pouco o resultado.

A raiz da covariância média é o **piso**: a volatilidade de uma carteira com infinitas ações parecidas com as nossas. É o risco de mercado.

> **O que esperar:** as duas volatilidades de referência e uma tabela em que as colunas "Típica" e "Fórmula" são iguais em todas as linhas.

In [ ]:
cov = retornos.cov() * 252
variancias = retornos.var() * 252
n = len(nomes)

variancia_media = variancias.mean()
covariancia_media = (cov.sum().sum() - variancias.sum()) / (n * (n - 1))

formula = []
for k in por_tamanho.index:
    formula.append(((1 / k) * variancia_media + (1 - 1 / k) * covariancia_media) ** 0.5 * 100)
por_tamanho["Fórmula (%)"] = formula

print(f"Volatilidade típica de uma ação sozinha: {variancia_media ** 0.5 * 100:.2f}%")
print(f"Piso do risco de mercado: {covariancia_media ** 0.5 * 100:.2f}%")
por_tamanho[["Típica (%)", "Fórmula (%)"]].round(4)

**Depois de rodar.** **[Preencher: o piso do risco de mercado.]**

### Bloco 14 · Volatilidade × número de ações

O gráfico resume a aula. No eixo horizontal, o número de ações; no vertical, a volatilidade ao ano.

- `ax.fill_between` pinta a faixa entre a melhor e a pior combinação de cada tamanho, em azul-claro.
- `ax.plot` desenha a média das combinações, em azul, com um ponto em cada tamanho (`marker="o"`), e a fórmula, em preto pontilhado.
- `ax.axhline` marca o piso do risco de mercado, em cinza.
- `ax.set_xticks(por_tamanho.index)` mostra os números de 1 a 10 no eixo horizontal.

> **O que esperar:** uma curva que desce rápido de 1 para 3 ou 4 ações e depois quase se achata, aproximando-se da linha cinza do piso. A faixa azul-clara mostra que, com poucas ações, a sorte na escolha pesa muito: a pior carteira de duas ações pode ser mais arriscada do que a melhor de uma.

In [ ]:
piso = covariancia_media ** 0.5 * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(
    por_tamanho.index, por_tamanho["Mínima (%)"], por_tamanho["Máxima (%)"],
    color="#2a78d6", alpha=0.15, label="Da melhor à pior combinação",
)
ax.plot(por_tamanho.index, por_tamanho["Média (%)"], color="#2a78d6", linewidth=2, marker="o", label="Média das combinações")
ax.plot(por_tamanho.index, por_tamanho["Fórmula (%)"], color="black", linestyle=":", linewidth=2, label="Fórmula")
ax.axhline(piso, color="gray", linewidth=0.8, label="Piso: risco de mercado")
ax.set_xticks(por_tamanho.index)
ax.set_xlabel("Número de ações na carteira")
ax.set_ylabel("Volatilidade ao ano (%)")
ax.set_title("Quanto risco cai a cada ação acrescentada")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

**Depois de rodar.** **[Preencher: a partir de quantas ações a curva praticamente para de cair.]**

## Parte 5 · Validação

### Bloco 15 · A matriz de correlação está certa?

Duas propriedades que toda matriz de correlação precisa ter:

1. **Diagonal igual a 1.** O laço percorre as ações e, com um `if`, marca `False` se alguma ficar diferente de 1. Arredondamos para 10 casas, por causa das pequenas imprecisões dos decimais.
2. **Simetria.** A matriz transposta, `corr.T`, troca linhas por colunas. Se a matriz é simétrica, ela é igual à própria transposta, e a maior diferença entre as duas é zero.

> **O que esperar:** `True` nos dois testes.

In [ ]:
diagonal_ok = True
for acao in nomes:
    if round(corr.loc[acao, acao], 10) != 1:
        diagonal_ok = False

print("Diagonal toda igual a 1?", diagonal_ok)
print("Matriz simétrica?", (corr - corr.T).abs().max().max() < 1e-12)

### Bloco 16 · A fórmula bate com as 1.023 carteiras?

No bloco 13, as colunas "Típica" e "Fórmula" pareceram iguais. Aqui, medimos a maior diferença entre elas. A fórmula não é uma aproximação: ela dá exatamente a variância média das carteiras de cada tamanho, porque cada ação e cada par aparecem o mesmo número de vezes no conjunto das combinações.

> **O que esperar:** `Maior diferença entre as combinações e a fórmula: 0.0 ponto percentual`.

In [ ]:
diferenca = (por_tamanho["Típica (%)"] - por_tamanho["Fórmula (%)"]).abs().max()
print("Maior diferença entre as combinações e a fórmula:", round(diferenca, 6), "ponto percentual")

### Bloco 17 · Com 10 ações, perto da aula 2?

Com as 10 ações, a volatilidade calculada aqui deve ficar perto da volatilidade da carteira da aula 2. Perto, mas não igual: aqui, os pesos voltam a 10% todo dia; na aula 2, a carteira nunca era rebalanceada, e os pesos derivavam, como vimos na aula 4.

> **O que esperar:** duas volatilidades próximas, com diferença de alguns décimos de ponto percentual.

In [ ]:
carteira = (precos / precos.iloc[0]).mean(axis=1) * CAPITAL
vol_aula2 = carteira.pct_change().dropna().std() * 252 ** 0.5

print(f"Pesos iguais todo dia (esta aula): {volatilidade_grupo(retornos, nomes) * 100:.2f}%")
print(f"Sem rebalancear (aula 2):          {vol_aula2 * 100:.2f}%")

**Depois de rodar.** **[Preencher: os dois valores.]**

## Parte 6 · Interpretação

### Bloco 18 · O que os números dizem

**[Texto a completar com os resultados reais.]**

A correlação média entre as 10 ações foi de **[x]**. O par que mais andou junto foi **[par]** (**[x]**), e o que menos andou junto foi **[par]** (**[x]**). Uma ação sozinha teve volatilidade típica de **[x]%** ao ano; com as 10, a carteira ficou em **[x]%**, perto do piso de **[x]%** do risco de mercado.

Perguntas que orientam a interpretação:

1. **Quanto da redução de risco já aparece com poucas ações?** Compare a queda de 1 para 3 ações com a queda de 5 para 10.
2. **A escolha das ações importa?** Com duas ou três ações, a distância entre a melhor e a pior combinação é grande. Com 10, desaparece.
3. **Os setores explicam as correlações?** Os pares mais correlacionados têm algo em comum: o mesmo setor, a exposição ao dólar, ao petróleo ou aos juros?
4. **Por que existe um piso?** Porque todas as ações brasileiras sentem os mesmos choques da economia. Para baixar o piso, é preciso acrescentar ativos que andam de outro jeito: renda fixa, ativos de outros países ou outras classes de ativos. A aula 8 testa um deles: o Bitcoin.

## Parte 7 · Limitações

### Bloco 19 · O que esta análise não mostra

- **A correlação muda com o tempo.** Em crises, as ações tendem a cair juntas, e a correlação sobe justamente quando a diversificação seria mais útil. A média de cinco anos esconde essas variações.
- **Correlação mede só relação linear.** Duas ações podem ter correlação baixa e, mesmo assim, despencar juntas nos dias extremos.
- **Pesos iguais com rebalanceamento diário.** É uma simplificação. Carteiras reais têm pesos que derivam e custos para rebalancear (aula 4).
- **Um único período e um único país.** Com outras ações, outro período ou outros mercados, os números seriam diferentes.
- **Os dados do Yahoo Finance continuam precisando de conferência,** como em todas as aulas.

## Parte 8 · Exercícios

### Bloco 20 · Para praticar

1. **Um fundo de índice na lista.** Acrescente `"BOVA11.SA"`, um fundo que replica o Ibovespa, à lista `ACOES` e rode o notebook de novo. Qual ação tem a maior correlação com ele? Por quê?
2. **Um fundo imobiliário.** Acrescente um fundo imobiliário negociado na bolsa, como `"HGLG11.SA"`, e veja a correlação dele com as ações. Ele diversifica mais ou menos que uma 11ª ação?
3. **A melhor carteira de três ações.** Modifique o bloco 12 para guardar também o grupo de cada volatilidade e descubra qual combinação de três ações teve a menor volatilidade. As ações escolhidas fazem sentido, olhando o mapa de calor?
4. **Correlação de retornos mensais.** Calcule a matriz com retornos mensais: `precos.resample("ME").last().pct_change().dropna().corr()`. Ela fica maior ou menor que a dos retornos diários?
5. **Dias de queda forte.** Separe os dias em que a média das 10 ações caiu mais de 2%, com `retornos[retornos.mean(axis=1) < -0.02]`, e calcule a correlação só nesses dias. Ela sobe ou desce em relação à do período todo?

## Resumo

### Bloco 21 · O que aprendemos

**Em finanças:**

- A correlação mede quanto duas ações andam juntas, de −1 a +1.
- Quanto menor a correlação entre os ativos, maior a redução de risco ao combiná-los.
- A diversificação elimina o risco específico de cada empresa, mas não o risco de mercado.
- A volatilidade cai rápido nas primeiras ações e depois se aproxima de um piso: a covariância média.
- Com poucas ações, a sorte na escolha pesa muito; com mais ações, os resultados ficam parecidos.

**Em Python:**

- `corr()` e `cov()` calculam as matrizes de correlação e de covariância de uma tabela inteira.
- `plt.subplots` cria a figura e o eixo; `imshow` desenha uma tabela como mapa de cores.
- `range` gera sequências de números; um laço dentro de outro percorre linhas e colunas.
- `combinations` lista todas as combinações de um grupo; cada uma vem como uma tupla.
- `from biblioteca import ferramenta` importa só uma parte de uma biblioteca.

## Gancho para a aula 6

### Bloco 22 · Próxima aula

Até aqui, todas as carteiras tiveram pesos iguais: 10% em cada ação. Mas, se a correlação muda o risco, talvez outra divisão seja melhor. Mais peso nas ações que andam menos juntas? Menos peso nas mais voláteis?

Na próxima aula, vamos sortear milhares de combinações de pesos, desenhar a famosa fronteira eficiente e encontrar a carteira de menor risco e a de melhor Sharpe. E vamos fazer o teste mais importante: essas carteiras "ótimas" continuam boas no período seguinte? O Python ganha o NumPy, a biblioteca de contas com muitos números de uma vez.

## Glossário da aula

| Termo | Significado |
|---|---|
| Covariância | Medida de quanto duas séries variam juntas, na escala das próprias séries |
| Correlação | Covariância padronizada, entre −1 e +1 |
| Matriz de correlação | Tabela com a correlação entre todos os pares de ativos |
| Risco específico | Parte do risco que vem de cada empresa; diversificável |
| Risco de mercado | Parte do risco que atinge todas as empresas; não diversificável |
| Diversificação | Combinar ativos que não andam perfeitamente juntos para reduzir o risco |
| Mapa de calor | Gráfico que pinta cada célula de uma tabela de acordo com o valor |
| Tupla | Sequência de itens entre parênteses que não pode ser alterada |
| `combinations` | Ferramenta da `itertools` que lista todas as combinações de um grupo |
| `range` | Gera uma sequência de números inteiros |

---

*Python Finance Lab · Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*